# PARC2026 — 72d M3 batch probe controller

72a/72b/72cのDrive結果を読み、3モデルすべてがexact source/hash・1 optimizer step・effective batch 32でPASSしていることを確認します。成功しても出力は `READY_FOR_M3_RUNNER_IMPLEMENTATION` であり、M3 benchmark trainingは開始しません。


In [ ]:
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/parc2026')
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI_m3_probe'
PIN = '5baa5e3297ae476fe6f462ebcaac6d988f1a8e43'
URL = 'https://github.com/yu37330/py_AI.git'
if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f'Existing non-Git directory: {REPO}')
    subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', PIN], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', '--force', PIN], check=True)
got = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if got != PIN:
    raise RuntimeError(f'Repository pin mismatch: {got}')
print('72d controller code:', got, flush=True)
PROBE_DIR = Path('/content/drive/MyDrive/parc2026-cache/model-benchmark-v1/batch-probes')
OUT = Path('/content/drive/MyDrive/parc2026-cache/model-benchmark-v1/m3_batch_probe_summary.json')
subprocess.run([
    'python', '-u', str(REPO / 'tools/benchmark/validate_m3_batch_probes.py'),
    '--probe-dir', str(PROBE_DIR), '--out', str(OUT)
], cwd=str(REPO), check=True)
print('=== 72d COMPLETE ===', flush=True)
print('Next: implement guarded M3 runners. Benchmark training has NOT started.', flush=True)
